In [ ]:
import requests
import json
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StringType

# Config
APP_KEY = "<YOUR_TFL_APP_KEY>"

# Fetch Tube line status from TfL API
url = f"https://api.tfl.gov.uk/Line/Mode/tube/Status?app_key={APP_KEY}"
response = requests.get(url)
raw_data = response.json()

# Convert to JSON strings (handles nested structure)
json_strings = [json.dumps(record) for record in raw_data]

# Create DataFrame as raw JSON strings
df = spark.createDataFrame(json_strings, StringType()).toDF("raw_json")

# Add ingestion timestamp
df = df.withColumn("ingestion_timestamp", current_timestamp())

# Save to Bronze Lakehouse as Delta table
df.write.format("delta") \
    .mode("append") \
    .save("Tables/bronze_line_status")

print(f"✅ {df.count()} records ingested!")

StatementMeta(, 4409ba5a-38a3-47f5-a747-4fa01023b03f, 6, Finished, Available, Finished, False)

✅ 11 records ingested!


In [2]:
# Register the Delta table in the Lakehouse
spark.sql("""
    CREATE TABLE IF NOT EXISTS bronze_line_status
    USING DELTA
    LOCATION 'abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Bronze_LH.Lakehouse/Tables/bronze_line_status'
""")

print("✅ Table registered successfully!")

StatementMeta(, 90c34ef0-cc36-496a-817a-3750978a71df, 4, Finished, Available, Finished, False)

✅ Table registered successfully!
